#Filtering Population CSV File

In [ ]:
import pandas as pd

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load the data
file_path = '/content/drive/My Drive/Pop_Data_85-22.csv'  # Adjust if your file is nested inside folders
df = pd.read_csv(file_path)

# Clean up column names and values
df.columns = df.columns.str.strip().str.upper()
df['COUNTY'] = df['COUNTY'].str.replace(' County', '', regex=False).str.strip()
df['YEAR'] = df['YEAR'].astype(int)
df['TOTAL POPULATION'] = df['TOTAL POPULATION'].astype(str).str.replace(',', '', regex=True).astype(int)

# Define target filters
target_counties = ['Douglas', 'Logan', 'Eagle']
target_years = [1987, 1992, 1997, 2002, 2007, 2012, 2017, 2022]

# Apply filters
filtered_df = df[df['COUNTY'].isin(target_counties) & df['YEAR'].isin(target_years)]

# Output path
output_path = '/content/drive/My Drive/Filtered_Pop_3Counties.csv'
filtered_df.to_csv(output_path, index=False)

print(f"✅ Filtered CSV created at:\n{output_path}")


Mounted at /content/drive
✅ Filtered CSV created at:
/content/drive/My Drive/Filtered_Pop_3Counties.csv


#Pearson Matrix Codes

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# File paths
pop_path = '/content/drive/My Drive/pop_pearson.csv'
ag_path = '/content/drive/My Drive/nia_pear.csv'
dis_path = '/content/drive/My Drive/dis_pearson.csv'

# Douglas Population
pop_df = pd.read_csv(pop_path)
pop_df = pop_df[pop_df['COUNTY'].str.contains('Douglas', case=False)].copy()
pop_df = pop_df[['YEAR', 'TOTAL POPULATION']].rename(columns={'TOTAL POPULATION': 'Douglas Population'})
pop_df['Douglas Population'] = pop_df['Douglas Population'].astype(int)

# Irrigated Agriculture Acreage
ag_df = pd.read_csv(ag_path)
ag_df = ag_df[['YEAR', 'DOUGLAS']].rename(columns={'DOUGLAS': 'Irrigated Agriculture Acreage'})

# South Platte River Discharge
dis_df = pd.read_csv(dis_path, header=None)
dis_df.columns = ['River', 'Year', 'Discharge_cfs', 'Drainage_Area_acres', 'Lat', 'Lon']
dis_df = dis_df[dis_df['River'].str.contains('South Platte', case=False)].copy()
dis_df = dis_df[['Year', 'Discharge_cfs']].rename(columns={'Discharge_cfs': 'South Platte River Discharge'})

# Merge and deduplicate
merged = (
    pop_df
    .merge(ag_df, on='YEAR', how='inner')
    .merge(dis_df, left_on='YEAR', right_on='Year')
    .drop(columns=['Year'])
    .drop_duplicates()
)

# Compute full Pearson correlation matrix
numeric_df = merged.drop(columns=['YEAR'])
cor_matrix = numeric_df.corr()

# Create interactive heatmap
fig = px.imshow(
    cor_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Douglas County Pearson Correlation Matrix (Full 9-Box)',
    labels=dict(color='Correlation Coefficient'),
    aspect='auto'
)
fig.update_layout(
    margin=dict(l=40, r=40, t=60, b=40),
    font=dict(family='Segoe UI', size=14),
    title_font=dict(size=20)
)

# Save and download
html_path = '/content/Douglas_Correlation_Matrix_TitleCase.html'
fig.write_html(html_path)

from google.colab import files
files.download(html_path)



Mounted at /content/drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load data
pop_path = '/content/drive/My Drive/pop_pearson.csv'
ag_path = '/content/drive/My Drive/nia_pear.csv'
dis_path = '/content/drive/My Drive/dis_pearson.csv'

# Douglas Population
pop_df = pd.read_csv(pop_path)
pop_df = pop_df[pop_df['COUNTY'].str.contains('Douglas', case=False)].copy()
pop_df = pop_df[['YEAR', 'TOTAL POPULATION']].rename(columns={'TOTAL POPULATION': 'Douglas Population'})
pop_df['Douglas Population'] = pop_df['Douglas Population'].astype(int)

# Irrigated Agriculture Acreage
ag_df = pd.read_csv(ag_path)
ag_df = ag_df[['YEAR', 'DOUGLAS']].rename(columns={'DOUGLAS': 'Irrigated Agriculture Acreage'})

# South Platte River Discharge
dis_df = pd.read_csv(dis_path, header=None)
dis_df.columns = ['River', 'Year', 'Discharge_cfs', 'Drainage_Area_acres', 'Lat', 'Lon']
dis_df = dis_df[dis_df['River'].str.contains('South Platte', case=False)].copy()
dis_df = dis_df[['Year', 'Discharge_cfs']].rename(columns={'Discharge_cfs': 'South Platte River Discharge'})

# Merge and deduplicate
merged = (
    pop_df
    .merge(ag_df, on='YEAR', how='inner')
    .merge(dis_df, left_on='YEAR', right_on='Year')
    .drop(columns=['Year'])
    .drop_duplicates()
)

# Correlation matrix and upper triangle
numeric_df = merged.drop(columns=['YEAR'])
cor_matrix = numeric_df.corr()
mask = np.triu(np.ones(cor_matrix.shape), k=1).astype(bool)
result_df = cor_matrix.where(mask).stack().reset_index()
result_df.columns = ['Variable 1', 'Variable 2', 'Pearson Coefficient']

# Styled HTML output
html_table = result_df.to_html(index=False, float_format="%.3f", classes="correlation-table")
styled_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Douglas County Pearson Correlation Table</title>
    <style>
        body {{
            font-family: 'Segoe UI', sans-serif;
            background-color: #f9f9fb;
            padding: 30px;
            color: #333;
        }}
        h2 {{
            text-align: center;
            color: #2c3e50;
        }}
        .correlation-table {{
            margin: 20px auto;
            border-collapse: collapse;
            width: 80%;
            box-shadow: 0 0 10px rgba(0,0,0,0.1);
        }}
        .correlation-table th, .correlation-table td {{
            padding: 12px 16px;
            border: 1px solid #ddd;
        }}
        .correlation-table th {{
            background-color: #34495e;
            color: #fff;
            text-transform: capitalize;
        }}
        .correlation-table tr:nth-child(even) {{
            background-color: #ecf0f1;
        }}
        .correlation-table tr:hover {{
            background-color: #d0e6f7;
        }}
    </style>
</head>
<body>
    <h2>Douglas County Pearson Correlation Table</h2>
    {html_table}
</body>
</html>
"""

# Save and download
html_path = '/content/Douglas_Correlation_Styled_TitleCase.html'
with open(html_path, 'w') as f:
    f.write(styled_html)

from google.colab import files
files.download(html_path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# File paths
pop_path = '/content/drive/My Drive/pop_pearson.csv'
ag_path = '/content/drive/My Drive/nia_pear.csv'
dis_path = '/content/drive/My Drive/dis_pearson.csv'
swe_path = '/content/drive/My Drive/Vail_SWE_Filtered.csv'

# Eagle Population
pop_df = pd.read_csv(pop_path)
pop_df = pop_df[pop_df['COUNTY'].str.contains('Eagle', case=False)].copy()
pop_df = pop_df[['YEAR', 'TOTAL POPULATION']].rename(columns={'TOTAL POPULATION': 'Eagle Population'})
pop_df['Eagle Population'] = pop_df['Eagle Population'].astype(int)

# Irrigated Agriculture Acreage
ag_df = pd.read_csv(ag_path)
ag_df = ag_df[['YEAR', 'EAGLE']].rename(columns={'EAGLE': 'Irrigated Agriculture Acreage'})

# Colorado River Discharge – Glenwood
dis_df = pd.read_csv(dis_path, header=None)
dis_df.columns = ['River', 'Year', 'Discharge CFS', 'Drainage Area Acres', 'Lat', 'Lon']
dis_df = dis_df[dis_df['River'].str.contains('Colorado River Glenwood', case=False)].copy()
dis_df = dis_df[['Year', 'Discharge CFS']].rename(columns={'Discharge CFS': 'Colorado River Discharge'})

# Vail Snow Water Equivalent
swe_df = pd.read_csv(swe_path)
swe_df = swe_df[['YEAR', 'SWE_AVERAGE']].rename(columns={'SWE_AVERAGE': 'Vail Snow Water Equivalent'})

# Merge datasets and clean
merged = (
    pop_df
    .merge(ag_df, on='YEAR', how='inner')
    .merge(dis_df, left_on='YEAR', right_on='Year')
    .merge(swe_df, on='YEAR', how='inner')
    .drop(columns=['Year'])
    .drop_duplicates()
)

# Compute 4×4 Pearson matrix
numeric_df = merged.drop(columns=['YEAR'])
cor_matrix = numeric_df.corr()

# Create modern heatmap
fig = px.imshow(
    cor_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Eagle County Pearson Correlation Matrix (With SWE)',
    labels=dict(color='Correlation Coefficient'),
    aspect='auto'
)
fig.update_layout(
    margin=dict(l=40, r=40, t=60, b=40),
    font=dict(family='Segoe UI', size=14),
    title_font=dict(size=20)
)

# Export and download
html_path = '/content/Eagle_Correlation_Matrix_Styled_16Box.html'
fig.write_html(html_path)

from google.colab import files
files.download(html_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#FIltered Table Codes

In [3]:
import pandas as pd
import numpy as np

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# File paths
pop_path = '/content/drive/My Drive/pop_pearson.csv'
ag_path = '/content/drive/My Drive/nia_pear.csv'
dis_path = '/content/drive/My Drive/dis_pearson.csv'
swe_path = '/content/drive/My Drive/Vail_SWE_Filtered.csv'

# Load Eagle Population
pop_df = pd.read_csv(pop_path)
pop_df = pop_df[pop_df['COUNTY'].str.contains('Eagle', case=False)].copy()
pop_df = pop_df[['YEAR', 'TOTAL POPULATION']].rename(columns={'TOTAL POPULATION': 'Eagle Population'})
pop_df['Eagle Population'] = pop_df['Eagle Population'].astype(int)

# Load Irrigated Agriculture
ag_df = pd.read_csv(ag_path)
ag_df = ag_df[['YEAR', 'EAGLE']].rename(columns={'EAGLE': 'Irrigated Agriculture Acreage'})

# Load Colorado River Discharge – Glenwood
dis_df = pd.read_csv(dis_path, header=None)
dis_df.columns = ['River', 'Year', 'Discharge CFS', 'Drainage Area Acres', 'Lat', 'Lon']
dis_df = dis_df[dis_df['River'].str.contains('Colorado River Glenwood', case=False)].copy()
dis_df = dis_df[['Year', 'Discharge CFS']].rename(columns={'Discharge CFS': 'Colorado River Discharge'})

# Load SWE using corrected column name
swe_df = pd.read_csv(swe_path)
swe_df = swe_df[['YEAR', 'SWE_AVERAGE']].rename(columns={'SWE_AVERAGE': 'Vail Snow Water Equivalent'})

# Merge all datasets
merged = (
    pop_df
    .merge(ag_df, on='YEAR', how='inner')
    .merge(dis_df, left_on='YEAR', right_on='Year')
    .merge(swe_df, on='YEAR', how='inner')
    .drop(columns=['Year'])
    .drop_duplicates()
)

# Compute correlation matrix and extract upper triangle
numeric_df = merged.drop(columns=['YEAR'])
cor_matrix = numeric_df.corr()
mask = np.triu(np.ones(cor_matrix.shape), k=1).astype(bool)
result_df = cor_matrix.where(mask).stack().reset_index()
result_df.columns = ['Variable 1', 'Variable 2', 'Pearson Coefficient']

# Styled HTML output
html_table = result_df.to_html(index=False, float_format="%.3f", classes="correlation-table")
styled_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Eagle County Pearson Correlation Table (With SWE)</title>
    <style>
        body {{
            font-family: 'Segoe UI', sans-serif;
            background-color: #f9f9fb;
            padding: 30px;
            color: #333;
        }}
        h2 {{
            text-align: center;
            color: #2c3e50;
        }}
        .correlation-table {{
            margin: 20px auto;
            border-collapse: collapse;
            width: 80%;
            box-shadow: 0 0 10px rgba(0,0,0,0.1);
        }}
        .correlation-table th, .correlation-table td {{
            padding: 12px 16px;
            border: 1px solid #ddd;
        }}
        .correlation-table th {{
            background-color: #34495e;
            color: #fff;
            text-transform: capitalize;
        }}
        .correlation-table tr:nth-child(even) {{
            background-color: #ecf0f1;
        }}
        .correlation-table tr:hover {{
            background-color: #d0e6f7;
        }}
    </style>
</head>
<body>
    <h2>Eagle County Pearson Correlation Table (With SWE)</h2>
    {html_table}
</body>
</html>
"""

# Save and download
html_path = '/content/Eagle_Correlation_Table_With_SWE.html'
with open(html_path, 'w') as f:
    f.write(styled_html)

from google.colab import files
files.download(html_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# File paths
pop_path = '/content/drive/My Drive/pop_pearson.csv'
ag_path = '/content/drive/My Drive/nia_pear.csv'
dis_path = '/content/drive/My Drive/dis_pearson.csv'

# Logan Population
pop_df = pd.read_csv(pop_path)
pop_df = pop_df[pop_df['COUNTY'].str.contains('Logan', case=False)].copy()
pop_df = pop_df[['YEAR', 'TOTAL POPULATION']].rename(columns={'TOTAL POPULATION': 'Logan Population'})
pop_df['Logan Population'] = pop_df['Logan Population'].astype(int)

# Irrigated Agriculture Acreage
ag_df = pd.read_csv(ag_path)
ag_df = ag_df[['YEAR', 'LOGAN']].rename(columns={'LOGAN': 'Irrigated Agriculture Acreage'})

# South Platte River Discharge
dis_df = pd.read_csv(dis_path, header=None)
dis_df.columns = ['River', 'Year', 'Discharge CFS', 'Drainage Area Acres', 'Lat', 'Lon']
dis_df = dis_df[dis_df['River'].str.contains('South Platte', case=False)].copy()
dis_df = dis_df[['Year', 'Discharge CFS']].rename(columns={'Discharge CFS': 'South Platte River Discharge'})

# Merge and clean
merged = (
    pop_df
    .merge(ag_df, on='YEAR', how='inner')
    .merge(dis_df, left_on='YEAR', right_on='Year')
    .drop(columns=['Year'])
    .drop_duplicates()
)

# Compute full Pearson correlation matrix
numeric_df = merged.drop(columns=['YEAR'])
cor_matrix = numeric_df.corr()

# Create styled heatmap
fig = px.imshow(
    cor_matrix,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    title='Logan County Pearson Correlation Matrix (Full 9-Box)',
    labels=dict(color='Correlation Coefficient'),
    aspect='auto'
)
fig.update_layout(
    margin=dict(l=40, r=40, t=60, b=40),
    font=dict(family='Segoe UI', size=14),
    title_font=dict(size=20)
)

# Save and download
html_path = '/content/Logan_Correlation_Matrix_Styled_9Box.html'
fig.write_html(html_path)

from google.colab import files
files.download(html_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# File paths
pop_path = '/content/drive/My Drive/pop_pearson.csv'
ag_path = '/content/drive/My Drive/nia_pear.csv'
dis_path = '/content/drive/My Drive/dis_pearson.csv'

# Logan Population
pop_df = pd.read_csv(pop_path)
pop_df = pop_df[pop_df['COUNTY'].str.contains('Logan', case=False)].copy()
pop_df = pop_df[['YEAR', 'TOTAL POPULATION']].rename(columns={'TOTAL POPULATION': 'Logan Population'})
pop_df['Logan Population'] = pop_df['Logan Population'].astype(int)

# Irrigated Agriculture Acreage
ag_df = pd.read_csv(ag_path)
ag_df = ag_df[['YEAR', 'LOGAN']].rename(columns={'LOGAN': 'Irrigated Agriculture Acreage'})

# South Platte River Discharge
dis_df = pd.read_csv(dis_path, header=None)
dis_df.columns = ['River', 'Year', 'Discharge CFS', 'Drainage Area Acres', 'Lat', 'Lon']
dis_df = dis_df[dis_df['River'].str.contains('South Platte', case=False)].copy()
dis_df = dis_df[['Year', 'Discharge CFS']].rename(columns={'Discharge CFS': 'South Platte River Discharge'})

# Merge and clean
merged = (
    pop_df
    .merge(ag_df, on='YEAR', how='inner')
    .merge(dis_df, left_on='YEAR', right_on='Year')
    .drop(columns=['Year'])
    .drop_duplicates()
)

# Compute unique correlations
numeric_df = merged.drop(columns=['YEAR'])
cor_matrix = numeric_df.corr()
mask = np.triu(np.ones(cor_matrix.shape), k=1).astype(bool)
result_df = cor_matrix.where(mask).stack().reset_index()
result_df.columns = ['Variable 1', 'Variable 2', 'Pearson Coefficient']

# HTML styling
html_table = result_df.to_html(index=False, float_format="%.3f", classes="correlation-table")
styled_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Logan County Pearson Correlation Table</title>
    <style>
        body {{
            font-family: 'Segoe UI', sans-serif;
            background-color: #f9f9fb;
            padding: 30px;
            color: #333;
        }}
        h2 {{
            text-align: center;
            color: #2c3e50;
        }}
        .correlation-table {{
            margin: 20px auto;
            border-collapse: collapse;
            width: 80%;
            box-shadow: 0 0 10px rgba(0,0,0,0.1);
        }}
        .correlation-table th, .correlation-table td {{
            padding: 12px 16px;
            border: 1px solid #ddd;
        }}
        .correlation-table th {{
            background-color: #34495e;
            color: #fff;
            text-transform: capitalize;
        }}
        .correlation-table tr:nth-child(even) {{
            background-color: #ecf0f1;
        }}
        .correlation-table tr:hover {{
            background-color: #d0e6f7;
        }}
    </style>
</head>
<body>
    <h2>Logan County Pearson Correlation Table</h2>
    {html_table}
</body>
</html>
"""

# Save and trigger download
html_path = '/content/Logan_Correlation_Table_Modern.html'
with open(html_path, 'w') as f:
    f.write(styled_html)

from google.colab import files
files.download(html_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>